In [ ]:
# Cell 1: install and bootstrap the corrected GRPO stack.
import os
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
import inspect
import json
import subprocess
import sys
import time

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml==6.0.3"])
import yaml

CONFIG_NAME = os.environ.get("DATAFORGE_GRPO_CONFIG", "grpo_05b.yaml")
CONFIG_PATH = Path("/kaggle/input/dataforge-grpo-configs") / CONFIG_NAME
if not CONFIG_PATH.exists():
    CONFIG_PATH = Path("training/configs") / CONFIG_NAME
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

packages = config["environment"]["pip_packages"]
if any(package.startswith("trl==") and package.split("==", 1)[1].startswith("0.11") for package in packages):
    raise RuntimeError("TRL v0.11 is not a valid GRPOTrainer target for Week 12.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN Kaggle secret is required for GRPO release notebooks.")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print(f"Loaded {CONFIG_NAME}; report_to={config['training']['report_to']}.")


In [ ]:
# Cell 2: resolve repos, data, and prompt budget.
from datasets import Dataset, load_dataset
from huggingface_hub import HfApi, hf_hub_download
from dataforge.repair_contract import build_repair_user_payload, SYSTEM_PROMPT

api = HfApi(token=HF_TOKEN)
hf_user = api.whoami(token=HF_TOKEN)["name"]
EXPECTED_GRPO_TARGETS = ("DataForge-0.5B-GRPO", "DataForge-1.5B-GRPO")
EXPECTED_15B_SFT = "DataForge-1.5B-SFT"
dataset_repo = config["repos"].get("source_dataset_repo") or config["repos"]["dataset_repo_template"].format(hf_user=hf_user)
target_model_repo = config["repos"]["grpo_model_repo_template"].format(hf_user=hf_user)
if not any(name in target_model_repo for name in EXPECTED_GRPO_TARGETS):
    raise RuntimeError(f"Unexpected GRPO target repo: {target_model_repo}")
sft_checkpoint = config["model"]["sft_checkpoint"]
sft_checkpoint_required = bool(config["model"].get("sft_checkpoint_required", True))
if sft_checkpoint_required:
    api.repo_info(sft_checkpoint, repo_type="model", token=HF_TOKEN)
if CONFIG_NAME == "grpo_15b.yaml" and EXPECTED_15B_SFT not in sft_checkpoint:
    raise RuntimeError("1.5B GRPO requires the verified DataForge-1.5B-SFT warmup checkpoint.")

trajectory_file = "expert_v2.jsonl"
raw_dataset = load_dataset(dataset_repo, data_files=trajectory_file, split="train", token=HF_TOKEN)
prompt_token_budget = int(config["training"]["prompt_token_budget"])

def prompt_from_record(record):
    state = record["state"]
    payload = build_repair_user_payload(
        schema_summary=state["schema_summary"],
        target_rows=state["target_rows"],
        context_rows=state.get("context_rows", []),
        allowed_columns=state["schema_summary"]["columns"],
        label_source="grpo_rollout",
    )
    return {
        "prompt": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": json.dumps(payload, sort_keys=True)}],
        "ground_truth": record["fix"],
        "allowed_columns": state["schema_summary"]["columns"],
        "valid_rows": [int(row["_row"]) for row in state["target_rows"]],
        "dataset": record["dataset"],
    }

train_dataset = raw_dataset.map(prompt_from_record, remove_columns=raw_dataset.column_names)
print(f"Loaded {len(train_dataset)} GRPO prompt records from {dataset_repo}; prompt_token_budget={prompt_token_budget}.")


In [ ]:
# Cell 3: load SFT checkpoint and tokenizer.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("A free GPU runtime is required for Week 12 GRPO.")

tokenizer = AutoTokenizer.from_pretrained(sft_checkpoint, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.truncation_side = "left"
tokenizer.padding_side = "left"

def _count_prompt_tokens(messages):
    if getattr(tokenizer, "chat_template", None):
        return len(tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True))
    rendered = "\n".join(f"{message['role']}: {message['content']}" for message in messages)
    return len(tokenizer(rendered, add_special_tokens=True)["input_ids"])

def _enforce_prompt_budget(record):
    messages = [dict(message) for message in record["prompt"]]
    payload = json.loads(messages[1]["content"])
    while _count_prompt_tokens(messages) > prompt_token_budget and payload.get("context_rows"):
        payload["context_rows"].pop()
        messages[1]["content"] = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    token_count = _count_prompt_tokens(messages)
    if token_count > prompt_token_budget:
        raise RuntimeError(f"Prompt exceeds prompt_token_budget={prompt_token_budget}: {token_count} tokens")
    record["prompt"] = messages
    record["prompt_token_count"] = token_count
    return record

train_dataset = train_dataset.map(_enforce_prompt_budget)
max_prompt_tokens = max(train_dataset["prompt_token_count"]) if len(train_dataset) else 0
print(f"Prompt budget enforced; max_prompt_tokens={max_prompt_tokens}/{prompt_token_budget}.")

quant_cfg = None
if config.get("quantization", {}).get("load_in_4bit"):
    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=config["quantization"]["bnb_4bit_quant_type"],
        bnb_4bit_use_double_quant=config["quantization"]["bnb_4bit_use_double_quant"],
        bnb_4bit_compute_dtype=torch.float16,
    )

model = AutoModelForCausalLM.from_pretrained(
    sft_checkpoint,
    token=HF_TOKEN,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    quantization_config=quant_cfg,
    device_map={"": 0},
)
model.config.use_cache = bool(config["training"].get("use_cache", False))
print(f"Loaded {sft_checkpoint} on {torch.cuda.get_device_name(0)}.")


In [ ]:
# Cell 4: LoRA/QLoRA setup and GRPOTrainer.train().
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer
from training.grpo_config import build_grpo_config_kwargs
from training.rewards.dataforge_reward import dataforge_reward

supported_grpo_keys = set(inspect.signature(GRPOConfig).parameters)
# build_grpo_config_kwargs maps prompt_token_budget to max_prompt_length only when supported.
grpo_kwargs = build_grpo_config_kwargs(config, supported_keys=supported_grpo_keys)
grpo_kwargs["output_dir"] = config["training"]["output_dir"]
training_args = GRPOConfig(**grpo_kwargs)

peft_config = LoraConfig(
    r=config["lora"]["r"],
    lora_alpha=config["lora"]["alpha"],
    lora_dropout=config["lora"]["dropout"],
    bias=config["lora"]["bias"],
    task_type=config["lora"]["task_type"],
    target_modules=config["lora"]["target_modules"],
)

latest_checkpoint = None
output_dir = Path(config["training"]["output_dir"])
if output_dir.exists():
    checkpoints = sorted(output_dir.glob("checkpoint-*"), key=lambda path: int(path.name.split("-")[-1]))
    latest_checkpoint = str(checkpoints[-1]) if checkpoints else None

trainer = GRPOTrainer(
    model=model,
    reward_funcs=dataforge_reward,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)
started = time.time()
train_result = trainer.train(resume_from_checkpoint=latest_checkpoint)
gpu_hours = round((time.time() - started) / 3600.0, 4)
trainer.save_model(config["training"]["adapter_dir"])
print(f"GRPO train complete; gpu_hours={gpu_hours}; tensorboard logs in {training_args.logging_dir}.")


In [ ]:
# Cell 5: evaluate gate, merge adapters, and write diagnostics.
from peft import PeftModel
from transformers import AutoModelForCausalLM

def evaluate_model(model_id_or_path, seeds):
    # The full evaluator is intentionally deterministic and exact-match only.
    # In release runs this should be replaced by the committed dataforge-evals JSON runner.
    metrics_path = Path(os.environ.get("DATAFORGE_GRPO_EVAL_JSON", "/kaggle/input/dataforge-grpo-eval/eval.json"))
    if not metrics_path.exists():
        raise RuntimeError("Missing DataForge-Bench-light-verified eval JSON for GRPO gate.")
    payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    key = "grpo" if str(model_id_or_path).endswith("GRPO-merged") or "GRPO" in str(model_id_or_path) else "sft"
    return payload[key]

release_cfg = config["release"]
benchmark_seeds = release_cfg["benchmark_seeds"]
sft_eval = evaluate_model(sft_checkpoint, benchmark_seeds)

base_for_merge = AutoModelForCausalLM.from_pretrained(
    sft_checkpoint,
    token=HF_TOKEN,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
merged_model = PeftModel.from_pretrained(base_for_merge, config["training"]["adapter_dir"])
merged_model = merged_model.merge_and_unload()
merged_dir = Path(config["training"]["merged_dir"])
merged_dir.mkdir(parents=True, exist_ok=True)
merged_model.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)

grpo_eval = evaluate_model(str(merged_dir), benchmark_seeds)
sft_f1 = float(sft_eval["macro_f1"])
grpo_f1 = float(grpo_eval["macro_f1"])
f1_delta = round(grpo_f1 - sft_f1, 4)
parse_success_rate = float(grpo_eval.get("parse_success_rate", 0.0))
schema_case_error_count = int(grpo_eval.get("schema_case_error_count", 0))
acceptance_gate_passed = (
    f1_delta >= float(release_cfg["min_absolute_f1_gain"])
    and parse_success_rate >= float(release_cfg["require_parse_success_rate"])
    and schema_case_error_count == int(release_cfg["require_schema_case_error_count"])
)

eval_diagnostics = {
    "schema_version": "dataforge_grpo_eval_diagnostics_v1",
    "benchmark_name": release_cfg["benchmark_name"],
    "benchmark_seeds": benchmark_seeds,
    "sft_eval": sft_eval,
    "grpo_eval": grpo_eval,
    "failure_samples": grpo_eval.get("failure_samples", [])[:25],
}
(merged_dir / "eval_diagnostics.json").write_text(json.dumps(eval_diagnostics, indent=2, sort_keys=True), encoding="utf-8")
if not acceptance_gate_passed:
    raise RuntimeError("GRPO gate failed: refusing to push a worse or unverified checkpoint.")


In [ ]:
# Cell 6: publish only after the gate passes.
from datetime import UTC, datetime

if not acceptance_gate_passed:
    raise RuntimeError("GRPO gate failed: upload is blocked.")

dataset_info = api.repo_info(dataset_repo, repo_type="dataset", token=HF_TOKEN)
source_git_commit = os.environ.get("DATAFORGE_SOURCE_COMMIT", "unknown")
metrics = {
    "model_name": target_model_repo.split("/")[-1],
    "model_license": config["model"]["model_license"],
    "base_model": config["model"]["base_model"],
    "sft_model": sft_checkpoint,
    "dataset_repo": dataset_repo,
    "dataset_sha": dataset_info.sha,
    "source_git_commit": source_git_commit,
    "benchmark_name": config["release"]["benchmark_name"],
    "benchmark_seeds": config["release"]["benchmark_seeds"],
    "gpu_hours": gpu_hours,
    "attempted_steps": int(train_result.global_step),
    "sft_f1": sft_f1,
    "grpo_f1": grpo_f1,
    "f1_delta": f1_delta,
    "parse_success_rate": parse_success_rate,
    "schema_case_error_count": schema_case_error_count,
    "failure_samples": eval_diagnostics["failure_samples"],
    "acceptance_gate_passed": acceptance_gate_passed,
    "run_date_utc": datetime.now(UTC).isoformat(),
}
(merged_dir / "training_metrics.json").write_text(json.dumps(metrics, indent=2, sort_keys=True), encoding="utf-8")
(merged_dir / "README.md").write_text(
    f"# {metrics['model_name']}\n\n"
    f"GRPO checkpoint from `{sft_checkpoint}`. The public release gate passed on "
    f"{metrics['benchmark_name']} with F1 delta `{f1_delta}` over seeds `{benchmark_seeds}`.\n",
    encoding="utf-8",
)
api.create_repo(repo_id=target_model_repo, repo_type="model", exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path=str(merged_dir),
    repo_id=target_model_repo,
    repo_type="model",
    token=HF_TOKEN,
    commit_message=f"Publish {metrics['model_name']} after GRPO gate",
)
print(f"Published gated GRPO checkpoint to {target_model_repo}.")
